# Prototipe model pertumbuhan lele v5

Notebook ini memakai dua CSV **raw-only** v5. Target model adalah bobot rata-rata pada sampling berikutnya. Prediktor menggabungkan bobot sampling terakhir, pakan dan mortalitas interval berikutnya, lama interval, serta ringkasan kondisi air pada interval tersebut.

> Dataset sintetis: hasil hanya untuk menguji alur modelling dan integrasi, bukan bukti akurasi lapangan. Validasi ditahan per siklus agar baris dari siklus yang sama tidak tersebar ke train dan test. Output prediksi hanya sampai sampling berikutnya; estimasi hari mencapai target di web perlu dinyatakan sementara dan dihitung terpisah.

Harga tidak dimodelkan di notebook ini.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
growth = pd.read_csv(DATA_DIR / "growth_management_lele_v5_raw.csv", parse_dates=["tanggal_waktu"])
env = pd.read_csv(DATA_DIR / "environment_lele_v5_raw.csv", parse_dates=["tanggal_waktu"])
print("Growth:", growth.shape, "Environment:", env.shape)
print("Siklus:", growth.siklus_id.nunique())


In [ ]:
env_features = ["suhu_c", "do_mgl", "ph_air", "amonia_mgN_L"]
feature_names = ["bobot_awal_g", "umur_awal_hari", "lama_interval_hari", "pakan_interval_g", "mortalitas_interval_ekor", "suhu_rerata_c", "do_rerata_mgl", "ph_rerata", "tan_n_rerata_mgL", "proporsi_do_di_bawah_4"]
rows = []
for sid, g in growth.sort_values("tanggal_waktu").groupby("siklus_id"):
    g = g.sort_values("tanggal_waktu").reset_index(drop=True)
    e = env[env.siklus_id == sid].sort_values("tanggal_waktu")
    # Fitur memakai interval selesai (a -> b); target adalah sampling sesudahnya (c).
    for i in range(len(g)-2):
        a, b, c = g.iloc[i], g.iloc[i+1], g.iloc[i+2]
        period = e[(e.tanggal_waktu > a.tanggal_waktu) & (e.tanggal_waktu <= b.tanggal_waktu)]
        if period.empty: continue
        days = (b.tanggal_waktu-a.tanggal_waktu).total_seconds()/86400
        vals = period[env_features].mean()
        rows.append({"siklus_id": sid, "kolam_id": b.kolam_id, "tanggal_target": c.tanggal_waktu,
            "bobot_target_g": c.bobot_rata_rata_sampel_g, "bobot_awal_g": b.bobot_rata_rata_sampel_g,
            "umur_awal_hari": b.umur_hari, "lama_interval_hari": days, "pakan_interval_g": b.pakan_interval_g,
            "mortalitas_interval_ekor": b.jumlah_ikan_mati_interval_ekor, "suhu_rerata_c": vals.suhu_c,
            "do_rerata_mgl": vals.do_mgl, "ph_rerata": vals.ph_air, "tan_n_rerata_mgL": vals.amonia_mgN_L,
            "proporsi_do_di_bawah_4": (period.do_mgl < 4).mean()})
data = pd.DataFrame(rows).dropna().sort_values("tanggal_target").reset_index(drop=True)
print("Contoh interval:", data.shape, "| siklus:", data.siklus_id.nunique())
data.head()


## Baseline dan validasi kelompok

Baseline persistence menganggap bobot sampling berikutnya sama dengan bobot terakhir. Ridge dipilih karena menghasilkan rumus ringkas yang bisa dikonversi ke PHP. Tiga siklus ber-ID terakhir dari setiap kolam dipakai sebagai holdout. Fitur hanya memakai interval yang telah selesai. Hasil tetap hanya uji awal karena data sintetis.

In [ ]:
cycles = sorted(data.siklus_id.unique())
test_cycles = set()
for pond, part in data[["kolam_id", "siklus_id"]].drop_duplicates().groupby("kolam_id"):
    test_cycles.update(sorted(part.siklus_id)[-3:])
train = data[~data.siklus_id.isin(test_cycles)]; test = data[data.siklus_id.isin(test_cycles)]
Xcols = feature_names

def fit_ridge(train_df, alpha=1.0):
    X = train_df[Xcols].to_numpy(float); y = train_df.bobot_target_g.to_numpy(float)
    mu, sd = X.mean(axis=0), X.std(axis=0); sd[sd == 0] = 1
    Z = (X-mu)/sd; A = np.column_stack([np.ones(len(Z)), Z])
    penalty = np.eye(A.shape[1])*alpha; penalty[0,0] = 0
    return mu, sd, np.linalg.solve(A.T@A + penalty, A.T@y)
def predict(df, params):
    mu, sd, beta = params
    return beta[0] + ((df[Xcols].to_numpy(float)-mu)/sd)@beta[1:]
def metrics(y, p):
    return {"MAE_g": float(np.mean(np.abs(y-p))), "RMSE_g": float(np.sqrt(np.mean((y-p)**2)))}
params = fit_ridge(train); pred = predict(test, params)
print("Holdout siklus:", sorted(test_cycles))
print("Baseline persistence:", metrics(test.bobot_target_g, test.bobot_awal_g))
print("Ridge:", metrics(test.bobot_target_g, pred))


## Keluaran pertumbuhan untuk halaman panen

Model memperkirakan bobot pada sampling berikutnya memakai bobot terbaru dan kondisi lingkungan/manajemen interval sebelumnya yang telah teramati. ETA mencapai target belum dimodelkan; diperlukan pengujian multi-langkah dan pengamatan lapangan. Harga dan nilai jual belum dihitung.

In [ ]:
# Prediksi satu sampling ke depan berdasarkan fitur interval terakhir yang sudah lengkap.
latest_feature_rows = data.sort_values("tanggal_target").groupby("siklus_id").tail(1).copy()
latest_feature_rows["prediksi_bobot_sampling_berikutnya_g"] = predict(latest_feature_rows, params)
latest_feature_rows[["siklus_id", "tanggal_target", "bobot_awal_g", "prediksi_bobot_sampling_berikutnya_g"]].head()


## Ekspor koefisien Ridge untuk implementasi PHP

Koefisien memakai fitur terstandar. PHP melakukan standardisasi dengan mean dan std pada JSON, lalu menghitung intercept + jumlah (koefisien × fitur standar).

In [ ]:
import json
final_params = fit_ridge(data); mu, sd, beta = final_params
model = {"model": "Ridge regression", "alpha": 1.0, "target": "bobot sampling berikutnya (g)",
 "features": {name: {"mean": float(m), "std": float(s), "coefficient": float(c)} for name,m,s,c in zip(Xcols,mu,sd,beta[1:])},
 "intercept": float(beta[0]), "validation": {"method": "holdout 3 siklus utuh per kolam", "held_out_cycles": sorted(test_cycles),
 "baseline_persistence": metrics(test.bobot_target_g,test.bobot_awal_g), "ridge": metrics(test.bobot_target_g,pred)},
 "warning": "Dilatih pada data sintetis v5; validasi lapangan belum dilakukan."}
Path("ridge_growth_v5_coefficients.json").write_text(json.dumps(model, indent=2), encoding="utf-8")
print("ridge_growth_v5_coefficients.json tersimpan")
